#### ***Query Optimization***



##### **Query Optimmization means improve the user's original query into a better query,then rerank can find better context.**

#### ***Why Query Optimization***

##### **The problem is that users don't always write retrieval-friendly queries.**

In [1]:
### load the Environment Variables

from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
### Check/Validate the path to load the files from the Document
import os

path = "../kubernetes"

if os.path.exists(path):
    print("Path is valid")
else:
    print("Path is not valid")

Path is valid


In [3]:
#load the documents using DirectoryLoader 

from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()

print("Number Of Documents:",len(documents))

C:\Users\mukko\AppData\Local\Temp\ipykernel_8944\894974917.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader,PyMuPDFLoader


Number Of Documents: 3983


In [4]:
### create a chunks using splitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 100
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 12694


In [5]:
###BM25 Retriever
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(chunks)

bm25_retriever.k=10

In [6]:
####Initalize the embedding model

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model = "BAAI/bge-large-en-v1.5",
    model_kwargs = {"device":"cpu"},
    encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
### load the vcctors from Chroma DB
from langchain_chroma import Chroma

vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag",
    embedding_function=embedding_model
)


In [8]:
#### retriever for similarity search.
vector_retriever = vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":10}
)

In [9]:
### Hybrid Search
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever,bm25_retriever],
    weights = [0.7,0.3]
)

In [10]:
### Lets test with hybrid retriever and how many candidates are retrieved
docs = hybrid_retriever.invoke("What is Kubernetes Deployment?")
print(len(docs))

20


In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [44]:
### Get top 5 final document after reranking

def retrieve_and_rerank(query, k=5):

    retrieved_docs = hybrid_retriever.invoke(query)


    pairs = [[query,doc.page_content] for doc in retrieved_docs]

    scores = reranker.predict(pairs)

    ##zip the scores and retrieved order the documents based on score from highest to lowest

    ranked_docs = sorted(zip(retrieved_docs,scores),key = lambda x:x[1],reverse=True)

    ## Get top k documents using ranked_docs and k value
    top_docs = [doc for (doc,score) in ranked_docs[:k]]

    #for i,(doc,score) in enumerate(ranked_docs[:k],1):
        #print(f"""
       #         {i}  | Page:{doc.metadata.get("page")} | Score:{score}
        #""")

    return top_docs

In [13]:
### Format without unnecessary context

def build_context(documents):

    context = ""

    for i,doc in enumerate(documents,start=1):
        source = doc.metadata.get("source")
        source = source.replace("\\","/")
        source = source.split("/")[-1]
        #print(source)
        context+=f"""
        
    Source {source}
    Page: {doc.metadata.get("page")}
    Content: {doc.page_content}
        """
    return context



In [14]:
### Design a prompt
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the context does not contain the answer, say:
   "I don't have information based on the provided documents."
4. Keep the answer concise and directly relevant.
5. List the sources and page numbers used at the very end of your response.
6. Format the sources strictly like this:
[Source: filename, Page: number]

Context:
{context}

Question:
{question}

Answer:
""")

In [15]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-safeguard-20b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B30A5FE510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B30A6094F0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
query = "What is Kubernetes Deployment?"

top_5_chunks = retrieve_and_rerank(query)

for i,chunk in enumerate(top_5_chunks,start=1):
    print(f"------ Document {i} --------")
    print("Page:",chunk.metadata.get("page"))

------ Document 1 --------
Page: 9
------ Document 2 --------
Page: 5
------ Document 3 --------
Page: 638
------ Document 4 --------
Page: 6
------ Document 5 --------
Page: 12


In [17]:
## Change the project name for Langsmith observability.
from langchain_core.tracers import LangChainTracer

custom_tracer = LangChainTracer(project_name="Generation Evaluation For Kubernetes RAG.")

In [18]:
config = {"run_name":"Generation Evaluation","callbacks":[custom_tracer]}

In [ ]:
def final_rag_response(query):

    top_5_docs = retrieve_and_rerank(query)

    context = build_context(top_5_docs)

    #print("Context:",context)

    messages = prompt.invoke({"question":query,"context":context})

    response = llm.invoke(messages,config=config).content

    return response

#### ***Build the Query Optimizer***

In [33]:
from langchain_core.prompts import ChatPromptTemplate

query_optimizer_prompt = ChatPromptTemplate.from_template("""
You are a query optimization component for a Kubernetes documentation RAG system.

Rewrite the user's query into a clear, specific search query
that will help retrieve relevant documents.

Rules:
- Preserve the original intent.
- Add important context when it is implied by the query.
- Use precise technical terminology.
- Do not answer the question.
- Do not add information that changes the user's intent.
- Return only the optimized query.

User query:
{query}
""")

In [34]:
query_optimizer = query_optimizer_prompt | llm

In [35]:
query = "What happens if one goes down?"

In [38]:
optimized_query = query_optimizer.invoke({"query":query})
print("Original Query:",query)
print("Optimized Query:",optimized_query.content)

Original Query: What happens if one goes down?
Optimized Query: Impact of a single node failure in a Kubernetes cluster.


In [39]:
queries = [
    "What is a Kubernetes Deployment?",
    "What is a Kubernetes Pod?",
    "What is a Kubernetes Service?",
    "What is a ReplicaSet?",
    "How does a Deployment manage ReplicaSets during an update?",
    "How does a Service identify which Pods should receive traffic?"
]
for query in queries:
    optimized_query = query_optimizer.invoke({"query":query})
    print("Original Query:",query)
    print("Optimized Query:",optimized_query.content)

Original Query: What is a Kubernetes Deployment?
Optimized Query: Kubernetes Deployment resource definition and purpose in Kubernetes cluster management.
Original Query: What is a Kubernetes Pod?
Optimized Query: Definition of a Kubernetes Pod and its components in Kubernetes architecture.
Original Query: What is a Kubernetes Service?
Optimized Query: Kubernetes Service definition and its role in cluster networking and load balancing


Original Query: What is a ReplicaSet?
Optimized Query: Kubernetes ReplicaSet definition and purpose
Original Query: How does a Deployment manage ReplicaSets during an update?
Optimized Query: Kubernetes Deployment rolling‑update process: how a Deployment creates, scales, and deletes ReplicaSets during an update.
Original Query: How does a Service identify which Pods should receive traffic?
Optimized Query: Kubernetes Service selector mechanism: how a Service matches pods to forward traffic to based on pod labels and endpoints.


#### ***Original vs Optimized Retrieval***

In [43]:
test_cases = [
    {
        "original": "What is a Kubernetes Deployment?",
        "optimized": "Kubernetes Deployment resource definition and purpose in Kubernetes cluster management."
    },
    {
        "original": "What is a Kubernetes Pod?",
        "optimized": "Definition of a Kubernetes Pod and its components in Kubernetes architecture."
    },
    {
        "original": "What is a Kubernetes Service?",
        "optimized": "Kubernetes Service definition and its role in cluster networking and load balancing."
    },
    {
        "original": "What is a ReplicaSet?",
        "optimized": "Kubernetes ReplicaSet definition and purpose."
    },
    {
        "original": "How does a Deployment manage ReplicaSets during an update?",
        "optimized": "Kubernetes Deployment rolling-update process: how a Deployment creates, scales, and deletes ReplicaSets during an update."
    },
    {
        "original": "How does a Service identify which Pods should receive traffic?",
        "optimized": "Kubernetes Service selector mechanism: how a Service matches Pods to forward traffic based on Pod labels and endpoints."
    }
]

for test_case in test_cases:
    original_query = test_case['original']
    optimized_query = test_case['optimized']
    print("="*60)
    print("Original Query:",original_query)
    print("="*60)
    original_response = final_rag_response(original_query)
    print("Original Response:",original_response)
    print("="*60)
    print("Optimized Query:",optimized_query)
    print("="*60)
    optimized_response = final_rag_response(optimized_query)
    print("Optimized Response:",optimized_response)
    print("-"*100)

Original Query: What is a Kubernetes Deployment?

                1  | Page:9 | Score:8.065120697021484
        

                2  | Page:5 | Score:7.92137336730957
        

                3  | Page:2 | Score:7.556789398193359
        

                4  | Page:638 | Score:7.125511169433594
        

                5  | Page:6 | Score:6.637397766113281
        
Original Response: A Kubernetes Deployment is an object that defines and manages a set of application Pods. It specifies the desired number of replicas, the container image, and other pod settings. The control plane continuously monitors the Pods, creating, updating, or replacing them to keep the actual state in line with the specified desired state, providing self‑healing and easy scaling of the application.  

Sources:  
[Source: Concepts.pdf, Page: 5]  
[Source: Tutorials.pdf, Page: 9]  
[Source: Tutorials.pdf, Page: 2]
Optimized Query: Kubernetes Deployment resource definition and purpose in Kubernetes cluster manageme

### RAG with larger, more realistic questions.

In [45]:
queries = [
    "How do Deployments, ReplicaSets, and Pods work together to maintain the desired number of application instances, and what happens when a Pod fails?",

    "How does a Kubernetes Deployment perform a rolling update, including how it creates and scales ReplicaSets and how it maintains application availability during the update?",

    "How does a Kubernetes Service discover Pods created by a Deployment, and how are label selectors, EndpointSlices, ClusterIP, and load balancing involved in routing traffic?",

    "What happens in Kubernetes when a Pod managed by a Deployment is deleted or fails, and which controllers are responsible for detecting the failure and creating a replacement?",

    "How do multiple containers within the same Pod communicate and share resources, and what networking, storage, and IPC mechanisms are available to them?",

    "Explain the complete relationship between a Deployment, ReplicaSet, Pod, and Service when deploying and exposing a scalable application in Kubernetes.",

    "How does Kubernetes continuously reconcile the desired state of an application with the actual state, using Deployments, ReplicaSets, Pods, and Services as examples?"
]

for query in queries:
    print("="*60)
    print("Query:",query)
    print("="*60)
    response = final_rag_response(query)
    print("Response:",response)

Query: How do Deployments, ReplicaSets, and Pods work together to maintain the desired number of application instances, and what happens when a Pod fails?
Response: Deployments describe the desired number of Pods and create a ReplicaSet that enforces that number.  
A ReplicaSet watches its Pods and, if any Pod is deleted or crashes (e.g., node failure or maintenance), it automatically creates a new Pod to replace it, keeping the replica count stable.  
Thus, the Deployment → ReplicaSet → Pods chain guarantees the specified number of application instances and automatically recovers from a Pod failure.  

[Source: Concepts.pdf, Page: 130]  
[Source: Concepts.pdf, Page: 156]  
[Source: Concepts.pdf, Page: 164]
Query: How does a Kubernetes Deployment perform a rolling update, including how it creates and scales ReplicaSets and how it maintains application availability during the update?
Response: A Kubernetes Deployment performs a rolling update by creating a new ReplicaSet for the updated

### **Based on few test Query Optimization is not required for our Rag System**